# Representation Demystifies Flat Minima
## Notebook 13: Hessian Geometry

Notebook 17 showed:

> Same function. Different sharpness.

This notebook explains the mathematical object behind that observation:

\[
H = \nabla^2 L(\theta)
\]

the Hessian.

We explore curvature, eigenvalues, flat directions, sharp directions,
and why Hessian geometry became central to the flat-minima literature.

## Output files

Running this notebook creates:

- figures/13_curvature_family.png
- figures/13_isotropic_bowl.png
- figures/13_anisotropic_bowl.png
- figures/13_eigenvector_directions.png
- figures/13_lambda_max_vs_curvature.png
- results/13_hessian_summary.csv
- results/13_hessian_summary.json
- downloads/13_hessian_geometry_outputs.zip

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
DOWNLOADS = ROOT / "downloads"

for path in [FIGURES, RESULTS, DOWNLOADS]:
    path.mkdir(parents=True, exist_ok=True)

## 1. Curvature family

Consider:

\[
L(x)=ax^2
\]

The Hessian is:

\[
H=2a
\]

Larger Hessian values correspond to steeper curvature.

In [ ]:
x = np.linspace(-1.5, 1.5, 500)

coefficients = [0.25, 1, 5, 25]

plt.figure(figsize=(8,5))

rows = []

for a in coefficients:
    y = a * x**2
    h = 2 * a

    rows.append({
        "a": a,
        "hessian": h
    })

    plt.plot(x, y, label=f"a={a}, H={h}")

plt.title("Curvature family")
plt.xlabel("x")
plt.ylabel("loss")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "13_curvature_family.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 2. Isotropic bowl

\[
L(x,y)=x^2+y^2
\]

Curvature is identical in every direction.

In [ ]:
x = np.linspace(-2, 2, 150)
y = np.linspace(-2, 2, 150)

X, Y = np.meshgrid(x, y)
Z = X**2 + Y**2

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(X, Y, Z)

ax.set_title("Isotropic bowl")

fig_path = FIGURES / "13_isotropic_bowl.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 3. Anisotropic bowl

\[
L(x,y)=10x^2+y^2
\]

One direction is much steeper than the other.

In [ ]:
Z = 10*X**2 + Y**2

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(X, Y, Z)

ax.set_title("Anisotropic bowl")

fig_path = FIGURES / "13_anisotropic_bowl.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 4. Hessian matrix

For:

\[
L(x,y)=10x^2+y^2
\]

the Hessian is:

\[
H=
\begin{bmatrix}
20 & 0 \\
0 & 2
\end{bmatrix}
\]

Eigenvalues reveal the curvature in each principal direction.

In [ ]:
H = np.array([
    [20, 0],
    [0, 2]
])

eigenvalues, eigenvectors = np.linalg.eig(H)

print("Eigenvalues:", eigenvalues)
print("Eigenvectors:")
print(eigenvectors)

## 5. Eigenvector directions

Contours make the principal directions easier to visualize.

In [ ]:
Z = 10*X**2 + Y**2

plt.figure(figsize=(7,7))

plt.contour(X, Y, Z, levels=20)

origin = np.array([0,0])

for i in range(2):
    vec = eigenvectors[:,i]

    plt.arrow(
        0,
        0,
        vec[0],
        vec[1],
        head_width=0.08,
        length_includes_head=True
    )

    plt.text(
        vec[0]*1.1,
        vec[1]*1.1,
        f"λ={eigenvalues[i]:.0f}"
    )

plt.axis("equal")
plt.title("Eigenvector directions")

fig_path = FIGURES / "13_eigenvector_directions.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 6. Largest eigenvalue

Many flat-minima papers use:

\[
\lambda_{max}(H)
\]

as a sharpness metric.

Larger values correspond to steeper curvature somewhere in parameter space.

In [ ]:
curvature_values = np.linspace(0.25, 25, 100)

lambda_max = 2 * curvature_values

plt.figure(figsize=(8,5))

plt.plot(curvature_values, lambda_max)

plt.xlabel("quadratic coefficient a")
plt.ylabel("largest eigenvalue")

plt.title("Largest eigenvalue versus curvature")

plt.grid(True)

fig_path = FIGURES / "13_lambda_max_vs_curvature.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 7. Save results

In [ ]:
summary = pd.DataFrame(rows)

csv_path = RESULTS / "13_hessian_summary.csv"
json_path = RESULTS / "13_hessian_summary.json"

summary.to_csv(csv_path, index=False)

with open(json_path, "w") as f:
    json.dump(
        {
            "largest_eigenvalue": float(max(eigenvalues)),
            "smallest_eigenvalue": float(min(eigenvalues)),
            "condition_number": float(max(eigenvalues)/min(eigenvalues)),
            "engineering_statement":
                "Hessians measure curvature. Curvature measures geometry. Geometry is not automatically computation."
        },
        f,
        indent=2
    )

print(csv_path)
print(json_path)

## Notebook takeaway

Flat-minima research often measures:

\[
\lambda_{max}(H)
\]

because it captures the sharpest local direction in parameter space.

Notebook 17 showed that equivalent computations can exhibit different
measured sharpness.

Therefore:

> Hessians measure curvature.
>
> Curvature measures geometry.
>
> Geometry is not automatically computation.

In [ ]:
output_files = [
    FIGURES / "13_curvature_family.png",
    FIGURES / "13_isotropic_bowl.png",
    FIGURES / "13_anisotropic_bowl.png",
    FIGURES / "13_eigenvector_directions.png",
    FIGURES / "13_lambda_max_vs_curvature.png",
    RESULTS / "13_hessian_summary.csv",
    RESULTS / "13_hessian_summary.json"
]

zip_path = DOWNLOADS / "13_hessian_geometry_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in output_files:
        if f.exists():
            z.write(f, arcname=f.relative_to(ROOT))

print("Created:", zip_path)

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))